In [ ]:
import pandas as pd
import pickle
import torch as ch
from preprocessing.geneva_stroke_unit_preprocessing.utils import create_ehr_case_identification_column

In [ ]:
predictions_path = '/Users/jk1/temp/opsum_end/testing/with_imaging/xgb_test_results/test_predictions.pkl'
test_data_path = '/Users/jk1/temp/opsum_end/preprocessing/with_imaging/gsu_Extraction_20220815_prepro_30012026_154047/splits/test_data_early_neurological_deterioration_ts0.8_rs42_ns5.pth'
eds_data = '/Users/jk1/stroke_datasets/Extraction_20220815/eds_j1.csv'

In [ ]:
n_timesteps = 72
threshold = 0.239

In [ ]:
with open(predictions_path, 'rb') as f:
    predictions = pickle.load(f)
y_test, y_prob = predictions

In [ ]:
X_test_raw, y_test_raw = ch.load(test_data_path)

In [ ]:
eds_df = pd.read_csv(eds_data, delimiter=';', encoding='utf-8',
                         dtype=str)
eds_df['case_admission_id'] = create_ehr_case_identification_column(eds_df)

In [ ]:
cids = X_test_raw[:,0,0,0]

In [ ]:
y_prob_matrix = y_prob.reshape(-1, n_timesteps)
y_test_matrix = y_test.reshape(-1, n_timesteps)

In [ ]:
y_pred_matrix = (y_prob_matrix >= threshold).astype(int)

In [ ]:
true_positives = ((y_test_matrix == 1) & (y_pred_matrix == 1))
false_positives = ((y_test_matrix == 0) & (y_pred_matrix == 1))
true_negatives = ((y_test_matrix == 0) & (y_pred_matrix == 0))
false_negatives = ((y_test_matrix == 1) & (y_pred_matrix == 0))

In [ ]:
n_true_positives_per_patient = true_positives.sum(axis=1)

In [ ]:
true_positives_per_patient_df = pd.DataFrame({
    'case_admission_id': cids,
    'n_true_positives': n_true_positives_per_patient
})

In [ ]:
cids_with_true_positives = true_positives_per_patient_df[true_positives_per_patient_df['n_true_positives'] > 0]['case_admission_id'].tolist()

In [ ]:
len(cids_with_true_positives)

In [ ]:
# ensure that cids_with_true_positives is a subset of the cids in the y_test_raw set
set(cids_with_true_positives).issubset(set(y_test_raw.case_admission_id))

In [ ]:
true_positive_df = y_test_raw[y_test_raw['case_admission_id'].isin(cids_with_true_positives)].copy()
true_positive_df = true_positive_df[['case_admission_id', 'patient_id', 'sample_date', 'relative_sample_date', 'value', 'min_nihss', 'delta_to_min',]]

In [ ]:
true_positive_df

In [ ]:
eds_df

In [ ]:
true_positive_df = true_positive_df.merge(eds_df[['case_admission_id', 'DOB', 'eds_final_id']], on='case_admission_id', how='left')

In [ ]:
true_positive_df.columns

In [ ]:
true_positive_df = true_positive_df[['case_admission_id', 'eds_final_id', 'patient_id', 'DOB', 'sample_date',
       'relative_sample_date', 'value', 'min_nihss', 'delta_to_min',]]

In [ ]:
# rename eds_final_id to EDS
true_positive_df.rename(columns={'eds_final_id': 'EDS'}, inplace=True)
# rename sample_date to END_date
true_positive_df.rename(columns={'sample_date': 'END_date'}, inplace=True)
# rename value to END_value
true_positive_df.rename(columns={'value': 'END_value'}, inplace=True)

In [ ]:
# sort by case_admission_id 
true_positive_df.sort_values(by='case_admission_id', inplace=True)

In [ ]:
# true_positive_df.to_csv('/Users/jk1/Downloads/end_true_positives_test_set_25022026.csv', index=False)